# kaggle-vllm 0.2.0 Qwen TP=2 sharded-state regression

Output-free focused regression for the existing `waqasm86/kaggle-vllm-models` artifact. It never saves or regenerates model shards. Prefer an attached Kaggle Input or already-populated local directory to avoid another multi-gigabyte download.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import tempfile

EXPECTED_SDK_VERSION = "0.2.0"
MODEL_REPO = "waqasm86/kaggle-vllm-models"
MODEL_REVISION = "08bb62d0b68d20062e9009a9769c0df53d3dae21"
ALLOW_HUB_DOWNLOAD = False  # Set True only when no attached/cached artifact exists and storage permits.
WORK = Path("/kaggle/working/kaggle-vllm-020-qwen-regression")
RUNTIME = WORK / "runtime"
STAGED = RUNTIME / "vllm-staged"
OVERLAY = RUNTIME / "vllm-runtime-overlay"
MANIFEST = RUNTIME / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")
WORK.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", f"kaggle-vllm[hub]=={EXPECTED_SDK_VERSION}"], check=True)
import kaggle_vllm
import torch
assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION
assert platform.python_version() == "3.12.13"
assert torch.__version__ == "2.10.0+cu128" and torch.version.cuda == "12.8"
assert torch.cuda.device_count() == 2
assert all(torch.cuda.get_device_name(i) == "Tesla T4" for i in range(2))
assert all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(2))
torch_before = {"version": torch.__version__, "cuda": torch.version.cuda, "path": str(Path(torch.__file__).resolve())}


## Strict immutable runtime bootstrap and activation

In [ ]:
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)
BOOTSTRAP = [
    "kaggle-vllm", "bootstrap", "--strict",
    "--staged", str(STAGED), "--overlay", str(OVERLAY),
    "--cache", str(CACHE), "--manifest", str(MANIFEST),
]
subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)
subprocess.run(BOOTSTRAP, check=True)
manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
assert manifest["wheel"]["sha256"] == "5a9bd710b8a19fdd23abb3442baad892da977466f996334decd533a225f5fd0c"
from kaggle_vllm import activate_runtime
assert activate_runtime(MANIFEST)
doctor = subprocess.run(["kaggle-vllm", "doctor", "--strict", "--json"], check=True, capture_output=True, text=True)
assert json.loads(doctor.stdout)["compatible"] is True
torch_after = {"version": torch.__version__, "cuda": torch.version.cuda, "path": str(Path(torch.__file__).resolve())}
assert torch_after == torch_before
print("Strict bootstrap, doctor, and Torch preservation: PASS")


## Resolve an existing immutable model directory

In [ ]:
explicit = os.environ.get("KAGGLE_VLLM_QWEN_PATH")
qwen_path = Path(explicit).resolve() if explicit else None
if qwen_path is None:
    attached = sorted(Path("/kaggle/input").rglob("model-rank-0-part-0.safetensors"))
    if attached:
        qwen_path = attached[0].parent.resolve()
if qwen_path is None and ALLOW_HUB_DOWNLOAD:
    from huggingface_hub import snapshot_download
    qwen_path = Path(snapshot_download(
        repo_id=MODEL_REPO, revision=MODEL_REVISION,
        local_dir=str(WORK / "model"),
    )).resolve()
if qwen_path is None:
    raise RuntimeError("Attach the published Qwen artifact, set KAGGLE_VLLM_QWEN_PATH, or explicitly allow the large Hub download.")
assert qwen_path.is_dir(), qwen_path
print("Using existing Qwen artifact:", qwen_path)


## Structural inspection, TP=2 identity, and topology rejection

In [ ]:
from kaggle_vllm import inspect_sharded_model
inspection = inspect_sharded_model(qwen_path, expected_tensor_parallel_size=2)
assert inspection.valid, inspection.to_dict()
assert inspection.rank_count == 2
expected_shards = {f"model-rank-{rank}-part-{part}.safetensors" for rank in (0, 1) for part in (0, 1)}
assert {item.name for item in inspection.shards} == expected_shards
wrong_topology = inspect_sharded_model(qwen_path, expected_tensor_parallel_size=1)
assert not wrong_topology.valid
assert any("tensor parallel size 1" in item for item in wrong_topology.topology_errors)
print(json.dumps(inspection.to_dict(), indent=2))


## Symlink rejection without touching the published artifact

In [ ]:
from kaggle_vllm.exceptions import ShardedModelError
fixture = Path(tempfile.mkdtemp(prefix="kaggle-vllm-symlink-test-", dir="/kaggle/working"))
try:
    outside = fixture / "outside.safetensors"
    outside.write_bytes(b"not-a-model")
    model_fixture = fixture / "model"
    model_fixture.mkdir()
    (model_fixture / "config.json").write_text("{}", encoding="utf-8")
    (model_fixture / "tokenizer_config.json").write_text("{}", encoding="utf-8")
    (model_fixture / "model-rank-0-part-0.safetensors").symlink_to(outside)
    try:
        inspect_sharded_model(model_fixture, expected_tensor_parallel_size=1)
    except ShardedModelError as error:
        assert "symlink" in str(error).casefold()
    else:
        raise AssertionError("symlinked shard was not rejected")
finally:
    shutil.rmtree(fixture)
print("Symlink protection: PASS")


## TP=2 load and short generation

In [ ]:
from kaggle_vllm import KaggleLLM
from vllm import SamplingParams
llm = KaggleLLM(
    model=str(qwen_path), load_format="sharded_state", tensor_parallel_size=2,
    dtype="float16", max_model_len=2048, gpu_memory_utilization=0.70,
    enforce_eager=True, disable_custom_all_reduce=True,
)
output = llm.generate(
    ["Explain tensor parallel inference in one concise sentence."],
    SamplingParams(temperature=0.0, max_tokens=48),
)
assert output and output[0].outputs and output[0].outputs[0].text
print(output[0].outputs[0].text)
print("Qwen TP=2 sharded_state load/generation: PASS")


## Result

The optional OpenAI endpoint may be tested in a separate clean process using the same conservative TP=2 arguments. It is not required by this focused structural/load regression because the final package acceptance notebook already covers the server path.

In [ ]:
result = {
    "status": "PASS", "sdk_version": kaggle_vllm.__version__,
    "model_repository": MODEL_REPO, "model_revision": MODEL_REVISION,
    "model_path": str(qwen_path), "rank_count": inspection.rank_count,
    "native_wheel": manifest["wheel"],
    "shards": sorted(item.name for item in inspection.shards),
    "checks": {"strict_bootstrap": True, "strict_doctor": True, "torch_preserved": True, "structural_inspection": True, "tp2_identity": True, "symlink_rejection": True, "topology_mismatch_rejection": True, "tp2_load": True, "short_generation": True},
}
(WORK / "qwen-regression-evidence.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result, indent=2))
print("FINAL QWEN TP=2 REGRESSION: PASS")
